# 案例：工具调用与外部信息获取

> Last updated: 2026-11-15

## 核心学习目标

本案例以酒店推荐任务为对象，学习如何将地图查询能力封装为智能体可以调用的工具，通过流式执行过程，可以观察模型何时生成工具调用、工具返回哪些候选酒店，以及模型如何结合用户偏好生成最终推荐。

## 案例设计

案例首先定义 `recommend_hotel` 工具，由该工具调用百度或高德地图 POI 接口查询酒店候选列表。随后使用 `create_agent` 创建带记忆的酒店推荐 Agent。最后通过流式消息输出工具调用事件和模型回答，从而区分候选检索与最终推荐两个阶段。

地图供应商适配位于 `map_api.py`，酒店搜索与结果标准化位于 `hotel_search.py`，模型创建位于 `llm.py`。本案例需要项目根目录 `.env` 中配置 `DASHSCOPE_API_KEY`，并至少配置 `BAIDU_MAP_AK` 或 `AMAP_KEY`。

**学习重点**：
1. 理解 LangChain 工具的参数定义、返回结构和文档字符串约束。
2. 掌握使用 `create_agent` 将大模型、工具、系统提示词和 Checkpointer 组合为 Agent。
3. 观察 Agent 自动生成工具调用参数并读取工具结果的过程。
4. 区分地图候选查询与模型最终推荐两类职责。
5. 理解流式执行中模型文本、工具调用和工具结果事件的差异。

## 0. 环境

本案例使用仓库根目录的共享 `.venv`。运行 Notebook 前，请在仓库根目录执行：

```bash
uv pip install -r Chapter-5/requirements.txt
```

环境变量统一从仓库根目录 `.env` 加载，不在章节目录维护局部 `.env`。

In [11]:
import sys
from datetime import datetime

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8")
        except Exception:
            pass

print("当前时间:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z"))

当前时间: 2026-11-15 12:13:21 +0800


## 1. 依赖与模块导入

导入 Agent、工具、Checkpointer、酒店搜索模块和统一模型入口。`hotel_search.py` 负责酒店候选查询，`map_api.py` 负责底层地图请求，`llm.py` 负责读取模型相关环境变量并创建模型。

In [12]:
import asyncio
import json

from typing import Any, Dict, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver


# Hotel search module (calls Baidu / Amap Place APIs through map_api)
from hotel_search import search_hotels
from llm import create_llm

## 2. 酒店查询与结果标准化

`fetch_hotels` 负责调用 `hotel_search.py` 中的酒店搜索接口，并把百度和高德两种来源统一为 Agent 可读取的酒店字段。发生接口错误时，工具会返回错误信息而不是伪造酒店数据。

In [13]:
async def fetch_hotels(city: str, keyword: str = "酒店", limit: int = 10) -> Dict[str, Any]:
    """Query hotel candidates through the map API.

    The hotel search module handles provider selection and normalization.
    Requires ``BAIDU_MAP_AK`` or ``AMAP_KEY`` in the project environment.

    Args:
        city: Target city or region.
        keyword: Map-searchable hotel keyword.
        limit: Maximum number of candidates to return.

    Returns:
        A dictionary produced from ``HotelSearchResult``.
    """
    res = await search_hotels(city, limit=limit, keyword=keyword)
    return res.model_dump()

## 3. 定义酒店推荐工具

`recommend_hotel` 使用 Pydantic 参数 Schema 和 `@tool` 声明工具能力。工具参数严格对应酒店搜索模块实际接受的输入：`city`、地图可检索的 `keyword` 和 `limit`。其他条件保留在 Agent 对话上下文中，用于读取候选后的最终选择。

In [14]:
class HotelSearchInput(BaseModel):
    """Input schema for the hotel search API."""

    city: str = Field(description="目标城市，必填。")
    keyword: str | None = Field(
        default=None,
        description="地图可检索词，例如区县、景点、地标、街道或酒店品牌。",
    )
    limit: int = Field(
        default=10,
        ge=1,
        le=20,
        description="返回酒店候选数量，范围 1 到 20。",
    )


@tool(args_schema=HotelSearchInput)
async def recommend_hotel(
    city: str,
    keyword: Optional[str] = None,
    limit: int = 10,
) -> Dict[str, Any]:
    """Query hotel candidates without producing a final recommendation.

    Args:
        city: Target city.
        keyword: Optional map-searchable keyword.
        limit: Maximum number of candidates to return.

    Returns:
        A dictionary with the candidate list and active data source.
    """

    search_key = keyword or "酒店"
    result = await fetch_hotels(city, search_key, limit)
    payload: Dict[str, Any] = {
        "city": city,
        "keyword": keyword,
        "limit": limit,
        "hotel_list": result["hotels"],
        "data_source": result["data_source"],
    }
    if result.get("search_query"):
        payload["search_query"] = result["search_query"]
    if result.get("error"):
        payload["error"] = result["error"]
        payload["message"] = result.get("message")
    return payload

## 4. 创建带记忆的 Agent

使用 `create_agent` 组合模型、`recommend_hotel` 工具、系统提示词和 `MemorySaver`。Checkpointer 通过 `thread_id` 维护同一会话的历史状态。

In [15]:
# Assemble the tool-calling agent with memory
llm = create_llm()
memory = MemorySaver()
agent = create_agent(
    llm,
    tools=[recommend_hotel],
    system_prompt="""你是酒店推荐助手，可以通过配置的工具 recommend_hotel 查询酒店。
规则：
1. 从用户输入的任务中提取 recommend_hotel需要的参数后执行调用。
2. 根据工具返回的 hotel_list，结合用户全部主观偏好和预算，由你挑选并只向用户推荐一家最合适的酒店。
3. 非酒店相关问题，回复：我只能协助酒店推荐。""",
    checkpointer=memory
)

## 5. 对比：直接调用工具

本单元不经过 Agent，直接调用 `recommend_hotel` 工具。工具只接收其底层酒店搜索 API 实际支持的参数，用于查询并返回酒店候选列表，不负责生成最终推荐话术。

In [16]:
direct_result = await recommend_hotel.ainvoke(
    {"city": "大同", "keyword": "近景区", "limit": 10}
)

print("搜索词:", direct_result.get("search_query"))
print("数据来源:", direct_result.get("data_source"))
print("候选数量:", len(direct_result.get("hotel_list", [])))

if direct_result.get("error"):
    print("查询失败:", direct_result.get("message") or direct_result["error"])
else:
    for index, hotel in enumerate(direct_result["hotel_list"], 1):
        location = hotel.get("district") or hotel.get("address")
        print(f"  {index}. {hotel.get('name')} | {location}")

搜索词: 近景区 酒店
数据来源: baidu_place_v2
候选数量: 6
  1. 浑源恒禄民宿 | 浑源县
  2. 浑源旨岭宜景酒店(北岳恒山景区真武庙店) | 浑源县
  3. 瑞福民宿(大同古城景区店) | 平城区
  4. 万瑞宾馆(古城景区店) | 浑源县
  5. 遇见·恒山(恒山北路分店) | 浑源县
  6. 魏都水世界 | 平城区


## 6. Agent 自动调用工具与事件输出

`run_hotel_agent` 使用 `astream_events(version="v2")` 获取 Agent 事件流。模型自行决定何时调用 `recommend_hotel` 以及传入什么参数；函数会把工具调用、工具参数、工具返回结果和模型最终回答分别标注后输出。

In [17]:
# Stream execution and listen for tool-call events
def _content_to_str(content: Any) -> str:
    """Convert LangChain message content into plain text."""

    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text") or ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts)
    return str(content) if content else ""


def _format_event_value(value: Any) -> str:
    """Format a tool input or output as readable JSON text."""

    if isinstance(value, ToolMessage):
        value = value.content
    elif hasattr(value, "content"):
        value = value.content

    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            return value
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


async def run_hotel_agent(user_query: str, thread_id: str = "demo_01"):
    """Stream the hotel agent and yield model text and tool events.

    Args:
        user_query: User request to send to the agent.
        thread_id: Checkpointer thread identifier for the conversation.

    Yields:
        Model text chunks and formatted tool-call event descriptions.
    """

    inputs = {"messages": [("user", user_query)]}
    config = {"configurable": {"thread_id": thread_id}}
    model_answer_started = False

    async for event in agent.astream_events(inputs, config, version="v2"):
        event_type = event["event"]
        if event_type == "on_chat_model_stream":
            chunk = event["data"].get("chunk")
            text = _content_to_str(getattr(chunk, "content", chunk))
            if text:
                if not model_answer_started:
                    yield "\n【模型最终回答】\n"
                    model_answer_started = True
                yield text
        elif event_type == "on_tool_start":
            yield f"\n【工具调用】{event.get('name')}\n"
            yield "【工具调用参数】\n" + _format_event_value(event["data"].get("input")) + "\n"
        elif event_type == "on_tool_end":
            yield "\n【工具返回结果】\n" + _format_event_value(event["data"].get("output")) + "\n"

## 7. 运行酒店推荐案例

默认示例问题为大同三天行程中的酒店推荐，需求包含安静和近景区两项偏好。运行后可以观察工具调用和最终推荐文本。

In [18]:
USER_QUERY = "我要去大同玩三天给我推荐酒店，需要安静/近景区"


async def main() -> None:
    """Run the default Chapter-5 hotel recommendation example."""

    print(f"用户: {USER_QUERY}\n")
    async for chunk in run_hotel_agent(USER_QUERY):
        print(chunk, end="", flush=True)
    print()

In [19]:
await main()

用户: 我要去大同玩三天给我推荐酒店，需要安静/近景区

模型回答: 
[调用工具：recommend_hotel]
[调用参数]
{
  "city": "大同",
  "limit": 20
}

[工具返回结果]
{
  "city": "大同",
  "keyword": null,
  "limit": 20,
  "hotel_list": [
    {
      "name": "天镇久天宾馆",
      "district": "天镇县",
      "address": "天镇第二小学西北约240米处",
      "tel": "13754900001",
      "location": "114.088393,40.423195",
      "rating": 5.0,
      "avg_price_cny": null,
      "type": "hotel"
    },
    {
      "name": "龙悦阁宾馆",
      "district": "云冈区",
      "address": "大同市恒安新区平盛街安宁里O区102号商铺",
      "tel": "(0352)4055444,19510319444",
      "location": "113.236434,40.022076",
      "rating": 5.0,
      "avg_price_cny": null,
      "type": "hotel"
    },
    {
      "name": "左云福源快捷宾馆",
      "district": "左云县",
      "address": "新东大街财政局对面",
      "tel": "03523831888",
      "location": "112.73158,40.020626",
      "rating": 5.0,
      "avg_price_cny": 85.0,
      "type": "hotel"
    },
    {
      "name": "大同优佳名宿",
      "district": "平城区",
      "address": "安益街益众园8号楼4-403

## 小结

1. **工具定义**将酒店候选查询封装为可被模型调用的结构化能力。
2. **Agent 组装**将模型、工具、系统提示词和会话记忆连接为执行图。
3. **工具参数**需要表达城市、偏好和预算等任务条件。
4. **流式执行**可以观察工具调用开始、工具返回结果和模型最终回答。
5. **候选检索与最终推荐分离**，使地图查询和语言决策各自承担清晰职责。

## 延伸思考：切换高德地图服务

本章前面的运行结果来自百度地图服务。`map_api.py` 已经封装了高德 Web 服务 API 的 `amap_place_text_search` 函数，`hotel_search.py` 在仅配置 `AMAP_KEY` 时会自动选择高德分支。请尝试仅配置 `AMAP_KEY`，用相同的城市与关键词重新运行本案例，并思考以下问题：

1. 针对相同的 query，高德地图搜索与百度地图搜索返回的原始字段内容有哪些差异？
2. 如果工具返回的字段存在差异，智能体如何稳定地调用工具并解析不同供应商的返回结果？
3. 回顾 `map_api.py` 与 `hotel_search.py`：统一酒店字段的适配发生在哪一层，它如何屏蔽供应商差异、使智能体不必感知底层服务的变化？